In [9]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

import torch.nn as nn
import pandas as pd
import numpy as np
from torchmetrics.classification import BinaryF1Score

from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score
from sklearn.preprocessing import RobustScaler
from torchmetrics.classification import BinaryF1Score, BinaryPrecision, BinaryRecall, BinaryAveragePrecision
from sklearn.metrics import confusion_matrix, roc_auc_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

True
NVIDIA GeForce RTX 5060 Laptop GPU
12.8
cuda


In [10]:
class GatedLayer(nn.Module):
    def __init__(self, input_size, output_size):
        super(GatedLayer, self).__init__()
        self.linear = nn.Linear(input_size, output_size)  # Linear transformation
        self.gate = nn.Linear(input_size, output_size)    # Gate transformation
        self.sigmoid = nn.Sigmoid()                       # Sigmoid activation for gate

    def forward(self, x):
        linear_output = self.linear(x)                    # Compute linear output
        gate_output = self.sigmoid(self.gate(x))         # Compute gate output and apply sigmoid
        return linear_output * gate_output  


#Architecture of the model
class CreditCardFraudModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Output layer with input size 8 and output size 1
        self.gated_layer = GatedLayer(29, 64)                 # Input layer with input size 29 (number of features) and output size 
        self.hidden_layer1 = nn.Linear(64, 32)
        self.hidden_layer2 = nn.Linear(32,16)                # First hidden layer with input size 32 and output size 16
        self.output_layer = nn.Linear(16, 1)     # Output layer with input size 16 and output size 1

        # Activation functions
        self.relu = nn.ReLU()
        
        self.dropout = nn.Dropout(p=0.3)  # Dropout layer with dropout probability of 0.3
        
        # Loss function and accuracy metric
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=device))
        self.accuracy_fn = BinaryF1Score()  
        self.batchnorm1 = nn.BatchNorm1d(64)  # Batch normalization for gated layer output (64 features)
        

    def forward(self, x):
        x = self.gated_layer(x)                  # Pass through gated layer
        x = self.batchnorm1(x)                 # Apply batch normalization
        x = self.relu(x)                       # Apply ReLU activation
        x = self.hidden_layer1(x)               # Pass through first hidden layer
        x = self.relu(x)                       # Apply ReLU activation
        x = self.hidden_layer2(x)              # Pass through second hidden layer
        x = self.relu(x)                       # Apply ReLU activation
        x = self.dropout(x)                    # Apply dropout
        x = self.output_layer(x)                # Pass through output layer
        return x                                # Return raw logits (no sigmoid)

In [11]:
import pickle

model = CreditCardFraudModel().to(device)
amount_scaler = RobustScaler()

# Load saved scaler (instead of refitting from CSV)
with open('Training_weights/amount_scaler.pkl', 'rb') as f:
    amount_scaler = pickle.load(f)
print("Scaler loaded from amount_scaler.pkl")


state_dict = torch.load("Training_weights/model_weights.pth", map_location=device)
model.load_state_dict(state_dict)
model.eval()


Scaler loaded from amount_scaler.pkl


CreditCardFraudModel(
  (gated_layer): GatedLayer(
    (linear): Linear(in_features=29, out_features=64, bias=True)
    (gate): Linear(in_features=29, out_features=64, bias=True)
    (sigmoid): Sigmoid()
  )
  (hidden_layer1): Linear(in_features=64, out_features=32, bias=True)
  (hidden_layer2): Linear(in_features=32, out_features=16, bias=True)
  (output_layer): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (loss_fn): BCEWithLogitsLoss()
  (accuracy_fn): BinaryF1Score()
  (batchnorm1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)

In [12]:
best_threshold = 0.95

test_file_path = 'test_PretrainedModel_creditcard.csv'
# Load test data with correct feature columns (including 'Amount')
test_df = pd.read_csv(test_file_path)
# Apply scaler fitted on training data
test_df.loc[:, 'Amount'] = amount_scaler.transform(test_df[['Amount']]).astype(np.float32).ravel()
# Select the same 29 features as in training
feature_columns = [f'V{i}' for i in range(1, 29)] + ['Amount']
test_inputs = test_df[feature_columns].values.astype(np.float32)
test_labels = test_df['Class'].values.astype(np.float32)

# Evaluate on test data using the best validation threshold.
test_inputs_manual = torch.as_tensor(test_inputs, dtype=torch.float32, device=device)
test_labels_manual = torch.as_tensor(test_labels, dtype=torch.float32, device=device).reshape(-1, 1)

model.eval()
with torch.no_grad():
    manual_logits = model(test_inputs_manual)

manual_probs = torch.sigmoid(manual_logits)
manual_true = test_labels_manual.int()

# Compute metrics using torchmetrics
manual_f1 = BinaryF1Score(threshold=best_threshold).to(device)(manual_probs, manual_true).item()
manual_precision = BinaryPrecision(threshold=best_threshold).to(device)(manual_probs, manual_true).item()
manual_recall = BinaryRecall(threshold=best_threshold).to(device)(manual_probs, manual_true).item()
manual_auprc = BinaryAveragePrecision().to(device)(manual_probs, manual_true).item()

# Compute confusion matrix and ROC AUC using sklearn
manual_true_cpu = manual_true.cpu().numpy().ravel()
manual_prob_cpu = manual_probs.cpu().numpy().ravel()
manual_pred_cpu = (manual_probs >= best_threshold).int().cpu().numpy().ravel()
manual_cm = confusion_matrix(manual_true_cpu, manual_pred_cpu)
manual_roc_auc = roc_auc_score(manual_true_cpu, manual_prob_cpu)

In [13]:
print(f"Best threshold: {best_threshold:.4f}")
print(f"  TN={manual_cm[0,0]:>6,}  FP={manual_cm[0,1]:>5,}")
print(f"  FN={manual_cm[1,0]:>6,}  TP={manual_cm[1,1]:>5,}")
print(f"  AUPRC    : {manual_auprc:.4f}")
print(f"  ROC AUC  : {manual_roc_auc:.4f}")
print(f"  F1 Score : {manual_f1:.4f}")
print(f"  Precision: {manual_precision:.4f}")
print(f"  Recall   : {manual_recall:.4f}")

Best threshold: 0.9500
  TN=28,425  FP=    7
  FN=     9  TP=   40
  AUPRC    : 0.8075
  ROC AUC  : 0.9805
  F1 Score : 0.8333
  Precision: 0.8511
  Recall   : 0.8163
